# Exercise 1 — compute_returns and compute_equity

Every backtest starts with a clean return series and an equity curve. `compute_returns` converts prices to daily percentage changes. `compute_equity` compounds those changes into a dollar curve that shows how $1 invested at the start would have grown (or shrunk).

In [ ]:
import pandas as pd, math

def _synthetic(n=252):
    prices = [100.0 * (1 + 0.3 * math.sin(i * 2 * math.pi / n)) for i in range(n)]
    dates  = pd.date_range("2023-01-01", periods=n, freq="B")
    close  = pd.Series(prices, index=dates)
    return pd.DataFrame({
        "Open":   close.shift(1).fillna(close.iloc[0]),
        "High":   close * 1.01,
        "Low":    close * 0.99,
        "Close":  close,
        "Volume": pd.Series([1_000_000 + i * 1_000 for i in range(n)], index=dates),
    })

def compute_returns(df):
    """Daily percentage returns from df["Close"].

    Returns pd.Series — same length as df.  First value is NaN.
    """
    # TODO: return df["Close"].pct_change()
    return pd.Series([float("nan")] * len(df), index=df.index)


def compute_equity(returns, initial=1.0):
    """Compound daily returns into an equity curve starting at `initial`.

    Treat NaN values as 0-return days (fillna(0) before compounding).
    Formula: (1 + returns.fillna(0)).cumprod() * initial
    """
    # TODO: return (1 + returns.fillna(0)).cumprod() * initial
    return pd.Series([initial] * len(returns), index=returns.index)


### Checks

In [ ]:
checks = 0

# 1 — compute_returns returns a Series of the same length
try:
    df = _synthetic()
    r  = compute_returns(df)
    assert isinstance(r, pd.Series) and len(r) == len(df)
    checks += 1; print("✅ 1 compute_returns returns same-length Series")
except Exception as e:
    print("❌ 1:", e)

# 2 — first value is NaN (no previous close)
try:
    df = _synthetic()
    r  = compute_returns(df)
    assert pd.isna(r.iloc[0]), "first return should be NaN"
    assert not pd.isna(r.iloc[1]), "second return should not be NaN"
    checks += 1; print("✅ 2 first return is NaN, second is not")
except Exception as e:
    print("❌ 2:", e)

# 3 — constant prices produce zero returns
try:
    df2 = _synthetic(); df2["Close"] = 50.0
    r   = compute_returns(df2)
    assert r.iloc[1:].abs().max() < 1e-9, f"expected zeros, got max {r.iloc[1:].abs().max()}"
    checks += 1; print("✅ 3 constant prices → zero returns")
except Exception as e:
    print("❌ 3:", e)

# 4 — compute_equity starts at `initial`
try:
    df = _synthetic()
    r  = compute_returns(df)
    eq = compute_equity(r, initial=1.0)
    assert isinstance(eq, pd.Series) and len(eq) == len(r)
    assert abs(eq.iloc[0] - 1.0) < 1e-9, f"equity[0] should be 1.0, got {eq.iloc[0]}"
    checks += 1; print("✅ 4 equity curve starts at initial=1.0")
except Exception as e:
    print("❌ 4:", e)

# 5 — all-positive returns → equity strictly greater than initial
try:
    pos_ret = pd.Series([0.01] * 30)
    eq = compute_equity(pos_ret, initial=1.0)
    assert (eq > 1.0).all(), f"expected all > 1.0, min={eq.min():.4f}"
    assert eq.iloc[-1] > eq.iloc[0], "equity should grow"
    checks += 1; print("✅ 5 all-positive returns → equity grows above initial")
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
